In [0]:
import requests
import time
from pyspark.sql.functions import current_timestamp

In [0]:
def get_weather(lat, lon):
    resp = requests.get(
        "https://api.open-meteo.com/v1/forecast",
        params={
            "latitude": lat, 
            "longitude": lon, 
            "current_weather": True},
        timeout=10,
    )
    resp.raise_for_status()
    w = resp.json()["current_weather"]
    return {
        "temperature_c": float(w["temperature"]),
        "windspeed_kmh": float(w["windspeed"]),
        "winddirection_deg": float(w["winddirection"]),
        "weathercode": int(w["weathercode"]),
        "is_day": int(w["is_day"]),
        "weather_time": w["time"],
    }

In [0]:
locations = spark.table("bronze.locations").collect()

In [0]:
weathers = []
for row in locations:
    try:
        weather = get_weather(row["lat"], row["long"])
        weathers.append({
            "ip": row["ip"],
            **weather,
        })
        print(f"{row['city']}, {row['country']}: {weather['temperature_c']}°C")
    except Exception as e:
        print(f"Skipping {row['city']}: {e}")
    time.sleep(1)



weather_df = spark.createDataFrame(weathers)

# add ingestion stamp
weather_df = weather_df.withColumn("ingested_at", current_timestamp())

# weather_df.display()

In [0]:
weather_df.write.mode("append").saveAsTable("bronze.weathers")